# Folder Structure Creation - Main

## Base Path

In [0]:
base_path = "/Volumes/weather_catalog/weather_platform/weather_data"

In [0]:
dbutils.fs.mkdirs(f"{base_path}/bronze")
dbutils.fs.mkdirs(f"{base_path}/silver")
dbutils.fs.mkdirs(f"{base_path}/gold")

display(dbutils.fs.ls(base_path))

## Open-Meteo Bronze source folder

In [0]:
dbutils.fs.mkdirs(f"{base_path}/bronze/open_meteo")

display(dbutils.fs.ls(f"{base_path}/bronze"))

## Chennai 2020 Bronze Folder (sample check)

In [0]:
dbutils.fs.mkdirs(
    f"{base_path}/bronze/open_meteo/city=chennai/year=2020"
)

display(
    dbutils.fs.ls(
        f"{base_path}/bronze/open_meteo/city=chennai"
    )
)

# Data Ingestion - Open Meteo

## Imports

In [0]:
import requests
import json

## Fixed Declaration

In [0]:
latitude = 13.0827
longitude = 80.2707

start_date = "2020-01-01"
end_date = "2020-12-31"

## API to pull data

In [0]:
url = (
    f"https://archive-api.open-meteo.com/v1/archive"
    f"?latitude={latitude}"
    f"&longitude={longitude}"
    f"&start_date={start_date}"
    f"&end_date={end_date}"
    f"&daily=temperature_2m_mean,precipitation_sum"
    f"&timezone=auto"
)

response = requests.get(url)

print(response.status_code)

## Check output

In [0]:
data = response.json()

print(data.keys())

In [0]:
print(data["daily"].keys())

## Save the data as raw

In [0]:
bronze_file_path = (
    f"{base_path}/bronze/open_meteo/"
    f"city=chennai/year=2020/raw.json"
)

dbutils.fs.put(
    bronze_file_path,
    json.dumps(data),
    overwrite=True
)

print("Raw JSON saved successfully")

## Check raw content

In [0]:
raw_content = dbutils.fs.head(
    f"{base_path}/bronze/open_meteo/city=chennai/year=2020/raw.json",
    500
)

print(raw_content)

## Get 5 years data

In [0]:
years = [2021, 2022, 2023, 2024]

for year in years:

    start_date = f"{year}-01-01"
    end_date = f"{year}-12-31"

    url = (
        f"https://archive-api.open-meteo.com/v1/archive"
        f"?latitude=13.0827"
        f"&longitude=80.2707"
        f"&start_date={start_date}"
        f"&end_date={end_date}"
        f"&daily=temperature_2m_mean,precipitation_sum"
        f"&timezone=auto"
    )

    response = requests.get(url)

    data = response.json()

    folder_path = (
        f"{base_path}/bronze/open_meteo/"
        f"city=chennai/year={year}"
    )

    dbutils.fs.mkdirs(folder_path)

    dbutils.fs.put(
        f"{folder_path}/raw.json",
        json.dumps(data),
        overwrite=True
    )

    print(f"Saved Chennai {year}")

## Check 5 years data

In [0]:
display(
    dbutils.fs.ls(
        f"{base_path}/bronze/open_meteo/city=chennai"
    )
)

## Check if raw files are present

In [0]:
for year in range(2020, 2025):
    path = f"{base_path}/bronze/open_meteo/city=chennai/year={year}"
    print(f"\n{path}")
    display(dbutils.fs.ls(path))

## 10 cities for easy looping

In [0]:
cities = {
    "chennai": (13.0827, 80.2707),
    "coimbatore": (11.0168, 76.9558),
    "mumbai": (19.0760, 72.8777),
    "delhi": (28.6139, 77.2090),
    "bangalore": (12.9716, 77.5946),
    "hyderabad": (17.3850, 78.4867),
    "kolkata": (22.5726, 88.3639),
    "pune": (18.5204, 73.8567),
    "ahmedabad": (23.0225, 72.5714),
    "kochi": (9.9312, 76.2673)
}

## Check city names

In [0]:
print(f"Total cities: {len(cities)}")

for city in cities:
    print(city)

## Ingesting data of 10 cities

In [0]:
years = [2020, 2021, 2022, 2023, 2024]

for city, (latitude, longitude) in cities.items():

    print(f"\nProcessing {city}...")

    for year in years:

        start_date = f"{year}-01-01"
        end_date = f"{year}-12-31"

        url = (
            f"https://archive-api.open-meteo.com/v1/archive"
            f"?latitude={latitude}"
            f"&longitude={longitude}"
            f"&start_date={start_date}"
            f"&end_date={end_date}"
            f"&daily=temperature_2m_mean,precipitation_sum"
            f"&timezone=auto"
        )

        response = requests.get(url)

        if response.status_code == 200:

            data = response.json()

            folder_path = (
                f"{base_path}/bronze/open_meteo/"
                f"city={city}/year={year}"
            )

            dbutils.fs.mkdirs(folder_path)

            dbutils.fs.put(
                f"{folder_path}/raw.json",
                json.dumps(data),
                overwrite=True
            )

            print(f"  Saved {city} - {year}")

        else:
            print(f"  Failed {city} - {year}")

## Verification - Open_Meteo

In [0]:
display(
    dbutils.fs.ls(
        f"{base_path}/bronze/open_meteo"
    )
)

# Data Ingestion - Meteostat

## Import meteostat

In [0]:
%pip install meteostat

In [0]:
import meteostat

print("Meteostat available")

## Meteostat Bronze source folder

In [0]:
dbutils.fs.mkdirs(f"{base_path}/bronze/meteostat")

display(dbutils.fs.ls(f"{base_path}/bronze"))

## Check meteostat

In [0]:
import meteostat

print(meteostat.__file__)
print(dir(meteostat))

In [0]:
import meteostat

print(meteostat.__version__)

In [0]:
import meteostat

print(dir(meteostat.daily))

## Imports

In [0]:
from meteostat import Point, daily
import pandas as pd
from datetime import datetime

## Check data

In [0]:
from meteostat import Point, daily
from datetime import datetime

chennai = Point(13.0827, 80.2707)

weather_data = daily(
    chennai,
    start=datetime(2020, 1, 1),
    end=datetime(2020, 12, 31)
)

df = weather_data.fetch()

print(df.shape)
print(df.columns)
df.head()

## Check error

In [0]:
chennai = Point(13.0827, 80.2707)

weather_data = daily(
    chennai,
    start=datetime(2020, 1, 1),
    end=datetime(2020, 12, 31)
)

print(type(weather_data))
print(weather_data)

## Meteostat error

- Attempted Meteostat as second source. 
- Library compatibility/data retrieval issues encountered in Databricks.
- Decided to use an alternative weather API for Source 2.

## Clear folder structure

In [0]:
dbutils.fs.rm(
    f"{base_path}/bronze/meteostat",
    recurse=True
)

## Check updated structure

In [0]:
display(dbutils.fs.ls(f"{base_path}/bronze"))

# Data Ingestion - Weather API

## Check historic data accessibility

In [0]:
api_key = "e9432c1b2db94e67a5873951261206"

url = (
    f"https://api.weatherapi.com/v1/history.json"
    f"?key={api_key}"
    f"&q=Chennai"
    f"&dt=2024-01-01"
)

response = requests.get(url)

print("Status Code:", response.status_code)

data = response.json()

print(data.keys())

## Check Endpoints

In [0]:
print(data["forecast"].keys())

In [0]:
print(json.dumps(data, indent=2)[:2000])

## Check 2020 data

In [0]:
import requests

url = (
    f"https://api.weatherapi.com/v1/history.json"
    f"?key={api_key}"
    f"&q=Chennai"
    f"&dt=2020-01-01"
)

response = requests.get(url)

print("Status Code:", response.status_code)

data_2020 = response.json()

print(data_2020)

- Switching to other source because it is returnig only 1 day per API call, so it will take more than 10k calls to extract 5 years data for 10 citites.


#Data Ingestion - NASA Power


## Checking for
- API reachable
- No API key needed
- One year returned
- Daily temperature available
- Daily precipitation available

In [0]:
url = """
https://power.larc.nasa.gov/api/temporal/daily/point
?parameters=T2M,PRECTOTCORR
&community=RE
&longitude=80.2707
&latitude=13.0827
&start=20200101
&end=20201231
&format=JSON
""".replace("\n","")

response = requests.get(url)

print("Status Code:", response.status_code)

data = response.json()

print(data.keys())

## To check whether required keys are available


In [0]:
print(data["properties"].keys())

In [0]:
print(data["properties"]["parameter"].keys())

In [0]:
print(list(data["properties"]["parameter"]["T2M"].items())[:5])

## Raw data verification

In [0]:
raw_path = f"{base_path}/bronze/nasa_power/city=chennai/year=2020"

dbutils.fs.mkdirs(raw_path)

dbutils.fs.put(
    f"{raw_path}/raw.json",
    json.dumps(data),
    overwrite=True
)

print("NASA POWER raw JSON saved")

## Verify raw file

In [0]:
display(
    dbutils.fs.ls(
        f"{base_path}/bronze/nasa_power/city=chennai/year=2020"
    )
)

## Check error - coords

In [0]:
print(type(cities))

for city, value in cities.items():
    print(city)
    print(value)
    break

## Save data for 5 years and 10 cities

In [0]:
for city, coords in cities.items():

    lat, lon = coords

    print(f"\nProcessing {city}...")

    for year in range(2020, 2025):

        start_date = f"{year}0101"
        end_date = f"{year}1231"

        url = (
            "https://power.larc.nasa.gov/api/temporal/daily/point"
            f"?parameters=T2M,PRECTOTCORR"
            f"&community=RE"
            f"&longitude={lon}"
            f"&latitude={lat}"
            f"&start={start_date}"
            f"&end={end_date}"
            f"&format=JSON"
        )

        response = requests.get(url)

        if response.status_code == 200:

            data = response.json()

            raw_path = (
                f"{base_path}/bronze/nasa_power/"
                f"city={city}/year={year}"
            )

            dbutils.fs.mkdirs(raw_path)

            dbutils.fs.put(
                f"{raw_path}/raw.json",
                json.dumps(data),
                overwrite=True
            )

            print(f"  Saved {city} - {year}")

        else:
            print(
                f"  Failed {city} - {year} "
                f"(Status: {response.status_code})"
            )

## Check whether data saved successfully


In [0]:
display(dbutils.fs.ls(f"{base_path}/bronze/nasa_power"))

# Final Check

## Open Meteo

In [0]:
display(dbutils.fs.ls(f"{base_path}/bronze/open_meteo"))

## NASA Power

In [0]:
display(dbutils.fs.ls(f"{base_path}/bronze/nasa_power"))